# Colab runbook: evaluate and improve the flame detector

Runs the repository's own scripts on a Colab GPU. Code comes from GitHub; the dataset, frozen
assets and run folders come from Google Drive. See `docs/SETUP.md` and `docs/PROJECT_STATUS.md`.

Rules (from the project handoff):
- Never evaluate on the held-out **test** split while tuning (`scripts/evaluate.py` does not allow it).
- Every changed experiment gets a **new config and a new output folder**; never resume `full_v1` with changes.
- Only one notebook may write to a run folder at a time.

Run with **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings
Edit these if your Drive layout differs.

In [ ]:
REPO_URL = 'https://github.com/Skandakm29/honeywell_bmsce_flame_detection.git'
BRANCH = 'feature/colab-eval-and-experiments'   # switch to 'main' once merged
DRIVE_ROOT = '/content/drive/MyDrive/HoneywellTraining'
HANDOFF_ZIP = f'{DRIVE_ROOT}/honeywell_training.zip'   # frozen manifests + TI initial weights
DATASET_ZIP = f'{DRIVE_ROOT}/FASDD_CV.zip'             # official archive, MD5 da5e1410e13113fc9d605567aea54508
RUNS = f'{DRIVE_ROOT}/runs'
REPO = '/content/honeywell'
PY = '/content/venv/bin/python'

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
for path in (HANDOFF_ZIP, DATASET_ZIP, f'{RUNS}/full_v1/best.pth'):
    print('found  ' if os.path.exists(path) else 'MISSING', path)

## 2. Code and environment
Pinned Torch 2.5.1/cu124 in an isolated venv, matching `full_v1`; the notebook kernel's own Torch is not touched.

In [ ]:
!rm -rf {REPO} && git clone -q --branch {BRANCH} {REPO_URL} {REPO} && git -C {REPO} log -1 --oneline
!pip install -q uv
!uv venv -q --python 3.12 /content/venv
!uv pip install -q --python {PY} torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu124
!uv pip install -q --python {PY} -r {REPO}/requirements.txt
!{PY} -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 3. Data
Copies the 12.3 GB archive to the VM disk (several minutes), restores the hash-checked frozen manifests, and extracts only the splits used here.

In [ ]:
LOCAL_ZIP = '/content/FASDD_CV.zip'
!cp -n "{DATASET_ZIP}" {LOCAL_ZIP} && ls -lh {LOCAL_ZIP}
!cd {REPO} && {PY} scripts/restore_assets.py --handoff "{HANDOFF_ZIP}"
!cd {REPO} && {PY} scripts/extract_local.py --archive {LOCAL_ZIP} --splits train val validation_full
!cd {REPO} && {PY} -m unittest discover -s tests

## 4. Baseline check
Re-evaluates `full_v1/best.pth` on the 2,000-image monitor split. It should reproduce the recorded
AP50:95 ≈ **0.3292** / AP50 ≈ **0.6269** (tiny GPU differences are fine). If it doesn't, stop and investigate.

In [ ]:
BASELINE = f'{RUNS}/full_v1'
!cd {REPO} && {PY} scripts/evaluate.py --checkpoint "{BASELINE}/best.pth" --split val --output "{BASELINE}/eval_val_check" --review 0

## 5. Full validation and failure review
All 26,765 cleaned validation images. Writes `summary.json` and review grids to Drive.

In [ ]:
EVAL = f'{BASELINE}/eval_validation_full'
!cd {REPO} && {PY} scripts/evaluate.py --checkpoint "{BASELINE}/best.pth" --split validation_full --output "{EVAL}"

In [ ]:
import json, glob
from IPython.display import Image, display
summary = json.load(open(f'{EVAL}/summary.json'))
print(json.dumps(summary['coco'], indent=2))
at = summary['at_confidence']
print(f"\nAt confidence {at['confidence']}: {at['misses']} missed flames, {at['false_detections']} false detections")
print('False detections by kind:', at['false_detections_by_kind'])
print('\nRecall by flame size (model-input pixels):')
for bucket, row in at['recall_by_input_size'].items():
    print(f"  {bucket:>8}: {row['detected']:>6}/{row['boxes']:<6} recall={row['recall'] if row['recall'] is None else round(row['recall'], 3)}")
print('\nBest F1 on full validation:', summary['threshold_sweep']['selected'])

In [ ]:
# Green = label, red = prediction at the operating confidence, yellow = the failure being reviewed.
for sheet in sorted(glob.glob(f'{EVAL}/review_misses_*.jpg'))[:2] + sorted(glob.glob(f'{EVAL}/review_false_detections_*.jpg'))[:2]:
    print(sheet); display(Image(sheet))

## 6. Next experiment
Pick **one** change based on step 5, give it a new config in `configs/` (committed via a PR) and a new name.
Re-running this cell after a disconnect resumes from `last.pth` in the same folder.

In [ ]:
CONFIG = 'configs/full_v1.json'   # replace with the new experiment's config
NAME = 'CHANGE_ME'                # e.g. v2_size512; must be new
assert NAME not in ('full_v1', 'CHANGE_ME'), 'Choose a new experiment name'
OUT = f'{RUNS}/{NAME}'
os.makedirs(OUT, exist_ok=True)
RESUME = f'--resume "{OUT}/last.pth"' if os.path.exists(f'{OUT}/last.pth') else ''
!cd {REPO} && {PY} scripts/train.py --config {CONFIG} --output "{OUT}" {RESUME} 2>&1 | tee -a "{OUT}/training.log"